In [43]:
from langchain_core.documents import Document

In [44]:
document = Document(
    page_content="This is the page content used for creating a document object in RAG pipeline",
    metadata={
        "author" : "Akshay Jain",
        "keywords" : ['RAG'],
        "page": 10,
        "timestamp" : "2026-01-01"
    }
)

### Loading files as Document Data Structure

https://docs.langchain.com/oss/python/integrations/document_loaders/pypdfdirectory

In [45]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

directory_path = "../data"

loader = PyPDFDirectoryLoader(path=directory_path)
docs = loader.load()

Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)


In [46]:
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-05-26T12:32:16+08:00', 'author': 'Simon Knollmeyer, Oğuz Caymazer and Daniel Grossmann', 'keywords': 'Retrieval-Augmented Generation; large language model; knowledge graph; production domain; manufacturing; design science research', 'moddate': '2025-05-26T07:13:00+02:00', 'subject': 'Retrieval-Augmented Generation (RAG) systems have shown significant potential for domain-specific Question Answering (QA) tasks, although persistent challenges in retrieval precision and context selection continue to hinder their effectiveness. This study introduces Document Graph RAG (GraphRAG), a novel framework that bolsters retrieval robustness and enhances answer generation by incorporating Knowledge Graphs (KGs) built upon a document’s intrinsic structure into the RAG pipeline. Through the application of the Design Science Research methodology, we systematically design, implement, and evaluate Gr

In [47]:
len(docs)

74

### Splitting Documents into chunks

LangChain text splitters are utilities used to break large text documents into smaller, manageable chunks that fit within a language model's (LLM) context window

This process, known as chunking, is crucial for maintaining context, improving retrieval-augmented generation (RAG), and working around token limits. 

Core Concepts
1. Chunk Size: The maximum length for each chunk (measured in characters or tokens).

2. Chunk Overlap: A specified amount of overlap between consecutive chunks to ensure continuity and prevent the loss of context at the boundaries of splits.

3. Separators: The characters or rules used to determine where to split the text

#### LangChain Text Splitters

LangChain offers various splitters tailored to different use cases and document formats:

*   **RecursiveCharacterTextSplitter**: The recommended default for generic text. It tries to split on a list of characters (e.g., `["\n\n", "\n", " ", ""]`) in order of priority, keeping semantically related pieces together as long as possible.
*   **CharacterTextSplitter**: A simple, length-based splitter that divides text by a single specified character (like a newline or space). It's fast but may cut sentences mid-way.
*   **TokenTextSplitter**: Splits text based on token counts rather than character counts, aligning with how LLMs interpret text. This ensures compatibility with specific model token limits.
*   **MarkdownHeaderTextSplitter**: Specifically designed for Markdown documents, it splits text based on structural elements like headings and subheadings, preserving the document's hierarchy and adding header information to the metadata of each chunk.
*   **HTML Splitters**: A suite of splitters for HTML content including `HTMLHeaderTextSplitter`, `HTMLSectionSplitter`, and `HTMLSemanticPreservingSplitter` that can split by header tags, specific sections, or preserve semantic elements like tables and lists.
*   **Specialized Splitters**: LangChain also provides splitters for specific languages and data types, including Python, Latex, JSON, NLTK, and spaCy.


#### Choosing and Using a Splitter

The optimal choice depends on your document type and task. 

For most general purposes, the `RecursiveCharacterTextSplitter` offers a good balance of context preservation and size management.

It tries to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

To use a splitter, you typically instantiate it with parameters like
- `chunk_size` 
- `chunk_overlap`

and then call the `split_text()` or `split_documents()` method


In [48]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

# Split a long text string
long_text = "..." # your large document
chunks = text_splitter.split_text(long_text)

In [49]:
### Text splitting get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [50]:
chunks=split_documents(docs)
chunks

Split 74 documents into 401 chunks

Example chunk:
Content: Academic Editor: Mohammad Shahin
Received: 14 March 2025
Revised: 13 May 2025
Accepted: 15 May 2025
Published: 22 May 2025
Citation: Knollmeyer, S.; Caymazer,
O.; Grossmann, D. Document
GraphRAG: Know...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-05-26T12:32:16+08:00', 'author': 'Simon Knollmeyer, Oğuz Caymazer and Daniel Grossmann', 'keywords': 'Retrieval-Augmented Generation; large language model; knowledge graph; production domain; manufacturing; design science research', 'moddate': '2025-05-26T07:13:00+02:00', 'subject': 'Retrieval-Augmented Generation (RAG) systems have shown significant potential for domain-specific Question Answering (QA) tasks, although persistent challenges in retrieval precision and context selection continue to hinder their effectiveness. This study introduces Document Graph RAG (GraphRAG), a novel framework that bolsters retrieval robustness and

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-05-26T12:32:16+08:00', 'author': 'Simon Knollmeyer, Oğuz Caymazer and Daniel Grossmann', 'keywords': 'Retrieval-Augmented Generation; large language model; knowledge graph; production domain; manufacturing; design science research', 'moddate': '2025-05-26T07:13:00+02:00', 'subject': 'Retrieval-Augmented Generation (RAG) systems have shown significant potential for domain-specific Question Answering (QA) tasks, although persistent challenges in retrieval precision and context selection continue to hinder their effectiveness. This study introduces Document Graph RAG (GraphRAG), a novel framework that bolsters retrieval robustness and enhances answer generation by incorporating Knowledge Graphs (KGs) built upon a document’s intrinsic structure into the RAG pipeline. Through the application of the Design Science Research methodology, we systematically design, implement, and evaluate Gr

### Create embeddings

For embeddings, 
- `sentence-transformers` open-source embedding model from HuggingFace

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2


For vectorDB
- `chromaDB` open-source

In [51]:
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Tuple, Dict, Any

In [ ]:
class EmbeddingManager:
    """Class to handle embedding generation using Sentence Transormaer"""

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        """Initialize embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence transformer
        """
        self.model_name = model_name
        self.model = None 

        try:
            print("Model loading started....")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the mode: {self.model_name}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings from a list of text

        Args: 
            text: List of texts

        Returns:
            Numpy array of embeddings. Shape (len(text), dimensions)
        """
        if self.model is None:
            raise ValueError("Model not launched yet.")
        
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings of shape ({len(texts)} ,{self.model.get_sentence_embedding_dimension()})")
        return embeddings
    

In [61]:
embedding_manager = EmbeddingManager()

Model loading started....


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 581.47it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding Dimensions: 384


In [62]:
chunk_text_list = [chunk.page_content for chunk in chunks]

embeddings = embedding_manager.generate_embeddings(chunk_text_list)

Generating embeddings for 401 texts


Batches: 100%|██████████| 13/13 [00:12<00:00,  1.02it/s]

Generated embeddings of shape (401 ,384)


### ChromaDB 

The chromadb.PersistentClient is a Python class used to create a local Chroma vector database instance that stores its data on disk, allowing data to persist across different sessions and program executions. 

#### Key Features & Usage
1. Data Persistence: Unlike the in-memory client, the PersistentClient automatically saves your database files (including a chroma.sqlite3 file) to a specified local directory.

2. Local Development: It is primarily intended for local development, testing, and embedded applications where setting up a separate Chroma server is not desired.

3. Simplicity and Latency: It offers a simple, low-latency solution as it runs within the same process as your application, avoiding network overhead. 

#### How to Use chromadb.PersistentClient
To use the persistent client, you need to import the chromadb library and instantiate the PersistentClient class, specifying a path where the data will be stored: 

In [ ]:
import chromadb

# Initialize the PersistentClient with a local path
client = chromadb.PersistentClient(path="./my_chroma_data")

# You can now interact with the client to create collections, add embeddings, and query data
collection = client.get_or_create_collection("my_collection")
collection.add(
    documents=["This is a document", "This is another document"],
    metadatas=[{"source": "doc1"}, {"source": "doc2"}],
    ids=["id1", "id2"]
)

#### Considerations


1. Not process-safe: The PersistentClient is thread-safe within a single process. Multiple clients cannot safely access the same persistence directory simultaneously from different processes.

2. Production Use: For production environments, the official documentation recommends using a server-backed Chroma instance with chromadb.HttpClient for better scalability and robustness.

https://docs.trychroma.com/docs/collections/add-data

In [63]:
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class VectorStore:
    """Manage storage of embeddings inside Vector Database (ChromaDB)"""

    def __init__(self, collection_name: str = "pdf_documents", persistent_directory_path : str = "../data/vector_store"):    
        """
        Initialize vector store

        Args:
            collection_name: Name of ChromaDB collection
            persistent_directory_path : path of local directory to store persisted data
        """
        self.collection_name = collection_name
        self.persistent_directory_path = persistent_directory_path
        self.client = None
        self.collection = None

        try:
            self.client = chromadb.PersistentClient(path=persistent_directory_path)
            self.collection = self.client.get_or_create_collection(
                name=collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """ 
        Add documents and embeddings to vector store

        Args:
            documents: List of Langchain Documents
            embeddings: ndarray
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chroma DB
        id_list = []
        embedding_list = []
        document_content_list = []
        metadata_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            id_list.append(doc_id) 

            # For embeddings
            embedding_list.append(emb.tolist())

            # For document
            document_content_list.append(doc.page_content)

            # For metadata
            metadata = dict(doc.metadata) # creating copy of metadata dict. Avoid modififying original, prevent side-effects
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadata_list.append(metadata)
        
        # Add to vector DB collection
        try:
            self.collection.add(
                ids=id_list,
                embeddings=embedding_list,
                metadatas=metadata_list,
                documents=document_content_list
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")


In [76]:
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [77]:
vector_store.add_documents(documents=chunks, embeddings=embeddings)

Adding 401 documents to vector store...
Successfully added 401 documents to vector store
Total documents in collection: 401


## RAG Retriver Pipeline

In [79]:
class RAGRetriever:
    """Handle query-based retrieval from vector store"""
    
    def __init__(self, embedding_manager: EmbeddingManager ,vector_store: VectorStore):
        """
        Initialize retriver object
        
        args:
            embedding_manager: EmbeddingManager object, which handles conversion of text to embedding
            vector_store: VectorStore object, storing embeddings
        """

        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, any]]:
        """
        Retrive relevant documents for a query

        Args:
            query: User query
            top_k: Top k closest results based on cosine similarity distance 
            score_threshold: Min. similarity score

        Returns:
            List of relevant documents as dict containing retrieved documents and metadata
        """

        # 1. Convert query to embeddings
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # 2. Get top k relevant documents from vector DB
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                # for 1st query
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0] # cosine similarity distances
                ids = results['ids'][0]

                # Filter results based on similarity score
                for i, (document, metadata, consine_distance, id) in enumerate(zip(documents, metadatas, distances, ids)):
                    similarity_score = 1 - consine_distance    
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'rank' : i + 1,
                            'id' : id,
                            'document' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : consine_distance,
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [80]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [82]:
query = "What is RAG"

rag_retriever.retrieve(query)

Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.09it/s]

Generated embeddings of shape (1 ,384)
Retrieved 5 documents (after filtering)


[{'rank': 1,
  'id': 'doc_8e946101_183',
  'document': 'could clarify its broader trajectory. This survey endeavors to\nfill this gap by mapping out the RAG process and charting\nits evolution and anticipated future paths, with a focus on the\nintegration of RAG within LLMs. This paper considers both\ntechnical paradigms and research methods, summarizing three\nmain research paradigms from over 100 RAG studies, and\nanalyzing key technologies in the core stages of “Retrieval,”\n“Generation,” and “Augmentation.” On the other hand, current\nresearch tends to focus more on methods, lacking analysis and\nsummarization of how to evaluate RAG. This paper compre-\nhensively reviews the downstream tasks, datasets, benchmarks,\nand evaluation methods applicable to RAG. Overall, this\npaper sets out to meticulously compile and categorize the\nfoundational technical concepts, historical progression, and\nthe spectrum of RAG methodologies and applications that\nhave emerged post-LLMs. It is design

## Connecting RAG Pipeline with LLM 